In [1]:
!python -V

Python 3.9.23


In [2]:
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import mean_squared_error


In [4]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

2025/07/29 06:07:32 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/07/29 06:07:32 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1', creation_time=1753506252722, experiment_id='1', last_update_time=1753506252722, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [5]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [6]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [7]:
len(df_train), len(df_val)

(73908, 61921)

In [8]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [9]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [10]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)


rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print("RMSE:", rmse)

RMSE: 7.758715209663881


In [12]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [13]:
with mlflow.start_run():

    mlflow.set_tag("developer", "Phan Dang Ha")

    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    # rmse = root_mean_squared_error(y_val, y_pred)
    # rmse = mean_squared_error(y_val, y_pred, squared=False)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin",artifact_path="models_pickle")

In [14]:
import xgboost as xgb
from xgboost import XGBRegressor


In [15]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [16]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [17]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=5000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=500
        )
        y_pred = booster.predict(valid)
        # rmse = root_mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [18]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': scope.int(hp.quniform('seed', 42, 62, 1))
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=Trials()
)

  0%|          | 0/10 [00:00<?, ?trial/s, best loss=?]

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:07:39] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.65031                          
[1]	validation-rmse:9.50082                           
[2]	validation-rmse:8.67353                           
[3]	validation-rmse:8.08633                           
[4]	validation-rmse:7.67448                           
[5]	validation-rmse:7.38636                           
[6]	validation-rmse:7.18564                           
[7]	validation-rmse:7.04508                           
[8]	validation-rmse:6.94469                           
[9]	validation-rmse:6.87462                           
[10]	validation-rmse:6.81947                          
[11]	validation-rmse:6.78384                          
[12]	validation-rmse:6.75855                          
[13]	validation-rmse:6.73882                          
[14]	validation-rmse:6.72071                          
[15]	validation-rmse:6.70770                          
[16]	validation-rmse:6.69838                          
[17]	validation-rmse:6.69196                          
[18]	valid

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:11:50] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.95606                                                   
[1]	validation-rmse:9.96174                                                    
[2]	validation-rmse:9.17611                                                    
[3]	validation-rmse:8.57747                                                    
[4]	validation-rmse:8.10994                                                    
[5]	validation-rmse:7.75147                                                    
[6]	validation-rmse:7.48166                                                    
[7]	validation-rmse:7.27044                                                    
[8]	validation-rmse:7.11531                                                    
[9]	validation-rmse:7.00017                                                    
[10]	validation-rmse:6.90837                                                   
[11]	validation-rmse:6.83685                                                   
[12]	validation-rmse:6.78140            

...

[0]	validation-rmse:11.75503                                                    
[1]	validation-rmse:11.32976                                                    
[2]	validation-rmse:10.93500                                                    
[3]	validation-rmse:10.57034                                                    
[4]	validation-rmse:10.23222                                                    
[5]	validation-rmse:9.91974                                                     
[6]	validation-rmse:9.63135                                                     
[7]	validation-rmse:9.36569                                                     
[8]	validation-rmse:9.12062                                                     
[9]	validation-rmse:8.89441                                                     
[10]	validation-rmse:8.68724                                                    
[11]	validation-rmse:8.49644                                                    
[12]	validation-rmse:8.32266

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:55:46] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.16133                                                    
[1]	validation-rmse:8.81858                                                     
[2]	validation-rmse:7.96143                                                     
[3]	validation-rmse:7.43110                                                     
[4]	validation-rmse:7.10603                                                     
[5]	validation-rmse:6.90676                                                     
[6]	validation-rmse:6.78205                                                     
[7]	validation-rmse:6.70341                                                     
[8]	validation-rmse:6.65041                                                     
[9]	validation-rmse:6.61646                                                     
[10]	validation-rmse:6.58996                                                    
[11]	validation-rmse:6.57263                                                    
[12]	validation-rmse:6.55795

/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [06:59:14] WARNING: /workspace/src/objective/regression_obj.cu:227: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:9.97846                                                     
[1]	validation-rmse:8.59978                                                     
[2]	validation-rmse:7.77949                                                     
[3]	validation-rmse:7.30489                                                     
[4]	validation-rmse:7.03162                                                     
[5]	validation-rmse:6.86941                                                     
[6]	validation-rmse:6.77472                                                     
[7]	validation-rmse:6.71547                                                     
[8]	validation-rmse:6.67817                                                     
[9]	validation-rmse:6.65151                                                     
[10]	validation-rmse:6.63302                                                    
[11]	validation-rmse:6.61865                                                    
[12]	validation-rmse:6.61247

In [19]:
mlflow.xgboost.autolog(disable=True)

In [20]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)
    
    best_params = {
        'learning_rate':    0.07411882787027682,
        'max_depth':    52,
        'min_child_weight':    1.752932154225768,
        'reg_alpha':    0.34374319303872397,
        'reg_lambda':   0.36034741418636235,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42
        
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=3000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=500    
    )

    y_pred = booster.predict(valid)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)
    
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")


[0]	validation-rmse:11.60932
[1]	validation-rmse:11.06352
[2]	validation-rmse:10.57127
[3]	validation-rmse:10.12688
[4]	validation-rmse:9.72761
[5]	validation-rmse:9.37046
[6]	validation-rmse:9.05010
[7]	validation-rmse:8.76468
[8]	validation-rmse:8.50901
[9]	validation-rmse:8.28249
[10]	validation-rmse:8.07875
[11]	validation-rmse:7.90209
[12]	validation-rmse:7.74392
[13]	validation-rmse:7.60497
[14]	validation-rmse:7.48226
[15]	validation-rmse:7.37192
[16]	validation-rmse:7.27374
[17]	validation-rmse:7.18807
[18]	validation-rmse:7.11140
[19]	validation-rmse:7.04431
[20]	validation-rmse:6.98507
[21]	validation-rmse:6.93092
[22]	validation-rmse:6.88433
[23]	validation-rmse:6.84138
[24]	validation-rmse:6.80457
[25]	validation-rmse:6.77083
[26]	validation-rmse:6.74078
[27]	validation-rmse:6.71476
[28]	validation-rmse:6.69030
[29]	validation-rmse:6.66919
[30]	validation-rmse:6.65032
[31]	validation-rmse:6.63294
[32]	validation-rmse:6.61671
[33]	validation-rmse:6.60295
[34]	validation-rmse

2025/07/29 07:08:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [07:08:59] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/07/29 07:09:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [21]:
from mlflow.models.signature import infer_signature

input_example = X_val[:5]  # hoặc dùng pd.DataFrame với 5 hàng đầu tiên
signature = infer_signature(X_val, y_pred[:5])


In [23]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)
    
    best_params = {
        'learning_rate':    0.07411882787027682,
        'max_depth':    52,
        'min_child_weight':    1.752932154225768,
        'reg_alpha':    0.34374319303872397,
        'reg_lambda':   0.36034741418636235,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'seed': 42
        
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=3000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=500    
    )

    y_pred = booster.predict(valid)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mlflow.log_metric("rmse", rmse)
    
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(
    booster,
    artifact_path="models_mlflow",
    signature=signature,
    input_example=input_example
    )



[0]	validation-rmse:11.60932
[1]	validation-rmse:11.06352
[2]	validation-rmse:10.57127
[3]	validation-rmse:10.12688
[4]	validation-rmse:9.72761
[5]	validation-rmse:9.37046
[6]	validation-rmse:9.05010
[7]	validation-rmse:8.76468
[8]	validation-rmse:8.50901
[9]	validation-rmse:8.28249
[10]	validation-rmse:8.07875
[11]	validation-rmse:7.90209
[12]	validation-rmse:7.74392
[13]	validation-rmse:7.60497
[14]	validation-rmse:7.48226
[15]	validation-rmse:7.37192
[16]	validation-rmse:7.27374
[17]	validation-rmse:7.18807
[18]	validation-rmse:7.11140
[19]	validation-rmse:7.04431
[20]	validation-rmse:6.98507
[21]	validation-rmse:6.93092
[22]	validation-rmse:6.88433
[23]	validation-rmse:6.84138
[24]	validation-rmse:6.80457
[25]	validation-rmse:6.77083
[26]	validation-rmse:6.74078
[27]	validation-rmse:6.71476
[28]	validation-rmse:6.69030
[29]	validation-rmse:6.66919
[30]	validation-rmse:6.65032
[31]	validation-rmse:6.63294
[32]	validation-rmse:6.61671
[33]	validation-rmse:6.60295
[34]	validation-rmse

2025/07/29 07:24:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
/opt/conda/envs/tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [07:24:32] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)


In [28]:
import mlflow
import pickle

run_id = "e1ff807e5a4b497bb75082d82427c22e"

# Tải file về local (hoặc cache tmp của MLflow)
path = mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/{run_id}/models_pickle/lin_reg.bin"
)

with open(path, "rb") as f_in:
    dv, model = pickle.load(f_in)  # ✅ dv trước, model sau

# ✅ Định nghĩa lại feature_cols (phải đúng như lúc huấn luyện)
feature_cols = ['PULocationID', 'DOLocationID', 'trip_distance']

# Tạo val_dicts nếu chưa có
val_dicts = df_val[feature_cols].to_dict(orient="records")

# Biến đổi và dự đoán
X_val_transformed = dv.transform(val_dicts)
predictions = model.predict(X_val_transformed)


In [29]:
loaded_model = mlflow.xgboost.load_model(logged_model)

In [30]:
y_pred = loaded_model.predict(xgb.DMatrix(X_val))

In [24]:
logged_model = 'runs:/6336a4572f82412b8fa1fd661c15f0c2/models_mlflow'


loaded_model = mlflow.pyfunc.load_model(logged_model)

# loaded_model.predict(pd.DataFrame())

In [25]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: /workspaces/mlops-zoomcamp/03-training/experiment_tracking/mlruns/1/models/m-8db8969a149c40d1bb7a6fe02fd9c140/artifacts
  flavor: mlflow.xgboost
  run_id: 6336a4572f82412b8fa1fd661c15f0c2